In [1]:
# # This Python 3 environment comes with many helpful analytics libraries installed
# # It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# # For example, here's several helpful packages to load

# import numpy as np # linear algebra
# import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# # Input data files are available in the read-only "../input/" directory
# # For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

# import os
# for dirname, _, filenames in os.walk('/kaggle/input'):
#     for filename in filenames:
#         print(os.path.join(dirname, filename))

# # You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# # You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
%%capture
!pip install transformers accelerate datasets peft bitsandbytes -q

In [3]:
import torch
import os
import pandas as pd
import numpy as np
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig,
    DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import matplotlib.pyplot as plt

In [4]:
# Set PyTorch memory management environment variable
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [5]:
# Load dataset
train_path = "/kaggle/input/hope-speech-detection-across-multiple-languages/Training_Data_PolyHope-M_2025/Training Data/en_train.csv"
dev_path = "/kaggle/input/hope-speech-detection-across-multiple-languages/Training_Data_PolyHope-M_2025/Training Data/en_dev.csv"

train_df = pd.read_csv(train_path)
dev_df = pd.read_csv(dev_path)

# Label encoding
label_mapping = {'Not Hope': 0, 'Generalized Hope': 1, 'Realistic Hope': 2, 'Unrealistic Hope': 3}
train_df['multiclass'] = train_df['multiclass'].map(label_mapping)
dev_df['multiclass'] = dev_df['multiclass'].map(label_mapping)

# Clean data
train_df.drop(columns=['binary'], inplace=True, errors='ignore')
dev_df.drop(columns=['binary'], inplace=True, errors='ignore')
train_df.dropna(inplace=True)
dev_df.dropna(inplace=True)

In [6]:
# Load tokenizer
base_model = "/kaggle/input/deepseek-r1/transformers/deepseek-r1-distill-llama-8b/2"
tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token

# Function to get token lengths
def get_token_lengths(texts):
    return [len(tokenizer.tokenize(text)) for text in texts]

# Compute token lengths and set max_length
train_token_lengths = get_token_lengths(train_df["text"])
dev_token_lengths = get_token_lengths(dev_df["text"])
max_length = min(int(np.percentile(train_token_lengths, 96)), 128)
print(f"Optimal max_length: {max_length}")

Optimal max_length: 77


In [7]:
# Prepare dataset with proper formatting
def prepare_dataset(df):
    texts = []
    labels = []
    for _, row in df.iterrows():
        text = f"Classify the following text: {row['text']}\nLabel:"
        texts.append(text)
        labels.append(row['multiclass'])
    
    dataset = Dataset.from_dict({"text": texts, "labels": labels})
    
    def tokenize_function(examples):
        tokenized = tokenizer(examples["text"], padding="max_length", truncation=True, max_length=max_length)
        tokenized["labels"] = [label if label != -100 else tokenizer.pad_token_id for label in examples["labels"]]
        return tokenized
    
    return dataset.map(tokenize_function, batched=True)

train_dataset = prepare_dataset(train_df)
dev_dataset = prepare_dataset(dev_df)

Map:   0%|          | 0/4541 [00:00<?, ? examples/s]

Map:   0%|          | 0/1650 [00:00<?, ? examples/s]

In [8]:
# Define quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load model
model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Prepare model for training
model = prepare_model_for_kbit_training(model)
model.config.use_cache = False
model.config.pretraining_tp = 1

# Define LoRA configuration with reduced parameters
peft_config = LoraConfig(
    r=8,  # Further reduced from 16
    lora_alpha=16,  # Further reduced from 32
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)

model = get_peft_model(model, peft_config)
model.print_trainable_parameters()

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

trainable params: 6,815,744 || all params: 8,037,076,992 || trainable%: 0.0848


In [9]:
# Custom compute_metrics function with memory-efficient evaluation
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    
    predictions = predictions.flatten()
    labels = labels.flatten()
    
    mask = labels != -100
    predictions = predictions[mask]
    labels = labels[mask]
    
    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0
    )
    return {
        "accuracy": accuracy,
        "f1": f1,
        "precision": precision,
        "recall": recall,
    }

In [10]:
# Training arguments with further memory optimizations
training_args = TrainingArguments(
    output_dir="./results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=2,  # Further reduced from 4
    per_device_eval_batch_size=2,  # Further reduced from 4
    num_train_epochs=8,
    weight_decay=0.01,
    save_total_limit=1,
    load_best_model_at_end=True,
    metric_for_best_model="eval_f1",
    greater_is_better=True,
    fp16=True,
    bf16=False,
    report_to="none",
    gradient_accumulation_steps=8,  # Increased from 4 (effective batch size = 2 * 8 = 16)
    optim="paged_adamw_8bit",
    warmup_ratio=0.1,
    lr_scheduler_type="cosine",
    logging_steps=10,
    max_grad_norm=0.3,
    gradient_checkpointing=True,
    eval_accumulation_steps=10,  # Added to move eval tensors to CPU in chunks
)

In [11]:
# Data collator
data_collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

# Trainer with EarlyStoppingCallback
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=dev_dataset,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(
        early_stopping_patience=3,
        early_stopping_threshold=0.01
    )]
)

In [12]:
# Train and save
trainer.train()
trainer.save_model("./deepseek-r1-classification")
tokenizer.save_pretrained("./deepseek-r1-classification")

Epoch,Training Loss,Validation Loss


In [ ]:
# Prediction function with batch processing
def predict(texts, model, tokenizer, max_length=128, batch_size=2):  # Reduced batch size
    device = model.device
    predictions = []
    
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i + batch_size]
        inputs = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors="pt",
        ).to(device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=1,
                pad_token_id=tokenizer.eos_token_id,
            )
        
        preds = outputs[:, -1].cpu().numpy()
        predictions.extend(preds)
        torch.cuda.empty_cache()  # Clear GPU memory after each batch
    
    return predictions

# Load test data
en_test = pd.read_csv('/kaggle/input/hope-speech-detection-across-multiple-languages/TestWithNoLabel/TestWithNoLabel/Ur_test_without_labels.csv')
test_texts = [f"Classify the following text: {text}\nLabel:" for text in en_test["text"].astype(str).tolist()]

# Make predictions
preds = predict(test_texts, model, tokenizer, max_length)

# Map predictions to labels
label_map = {0: "Not Hope", 1: "Generalized Hope", 2: "Realistic Hope", 3: "Unrealistic Hope"}
predicted_labels = [label_map[pred] for pred in preds]

# Save results
submission_df = pd.DataFrame({"Text": en_test["text"], "Tag": predicted_labels})
submission_df.to_csv("predictions.csv", index=False)
print("Predictions saved to predictions.csv")